<a href="https://colab.research.google.com/github/elKiruvi/Analisis-Sentimientos-BETO/blob/main/Analisis_Sentimientos_BETO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis de Sentimientos con BETO

## Parte 1: Dependencias, importaciones y configuraciones

In [ ]:
# ============================================================
# Instalacion de dependencias
# ============================================================
!pip install -q transformers datasets torch torchmetrics
!pip install -q scikit-learn matplotlib seaborn pandas numpy
!pip install -q accelerate -U

In [ ]:
# ============================================================
# Importaciones y configuracion global
# ============================================================
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import warnings
import transformers
import packaging.version as pv
import os
os.environ["TQDM_DISABLE"] = "1"

warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}  |  PyTorch {torch.__version__}  |  Transformers {transformers.__version__}")

## Parte 2: Carga, exploración y limpieza del datase

In [ ]:
# ============================================================
# Carga del dataset
# ============================================================
df = pd.read_csv("sentiment_analysis_dataset.csv")

TEXT_COL      = "text"
SENTIMENT_COL = "sentiment"

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
df.head()


In [ ]:
# ============================================================
# Analisis exploratorio
# ============================================================
print(df.info())
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")
print(f"\nDistribucion de sentimientos:\n{df[SENTIMENT_COL].value_counts()}")

df["tweet_length"] = df[TEXT_COL].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sentiment_counts = df[SENTIMENT_COL].value_counts()
axes[0].bar(sentiment_counts.index, sentiment_counts.values,
            color=sns.color_palette("husl", len(sentiment_counts)))
axes[0].set_title("Distribucion de sentimientos", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sentimiento")
axes[0].set_ylabel("Cantidad")
axes[0].tick_params(axis="x", rotation=45)

axes[1].hist(df["tweet_length"], bins=15, color="#4C72B0", edgecolor="white")
axes[1].set_title("Distribucion de longitud de tweets", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Caracteres")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig("distribucion_dataset.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nLongitud promedio: {df['tweet_length'].mean():.1f} | "
      f"Max: {df['tweet_length'].max()} | Min: {df['tweet_length'].min()}")

In [ ]:
# ============================================================
# Preprocesamiento de texto
# ============================================================
def limpiar_tweet(texto: str) -> str:
    """Elimina URLs, menciones y simbolos de hashtag. Conserva emojis."""
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"http\S+|www\S+|https\S+", "", texto, flags=re.MULTILINE)
    texto = re.sub(r"@\w+", "", texto)
    texto = re.sub(r"#(\w+)", r"\1", texto)
    texto = re.sub(r"[\x00-\x1F\x7F]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["text_clean"] = df[TEXT_COL].apply(limpiar_tweet)
df = df[df["text_clean"].str.len() > 3].reset_index(drop=True)
df = df.dropna(subset=[SENTIMENT_COL]).reset_index(drop=True)

In [ ]:
# ============================================================
# Codificacion de etiquetas
# ============================================================
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df[SENTIMENT_COL])

NUM_LABELS  = len(label_encoder.classes_)
LABEL_NAMES = list(label_encoder.classes_)

id2label    = {int(i): str(l) for i, l in enumerate(LABEL_NAMES)}
label2id    = {str(l): int(i) for i, l in enumerate(LABEL_NAMES)}

print(f"Clases detectadas ({NUM_LABELS}):")
for idx, name in id2label.items():
    count = (df["label"] == idx).sum()
    print(f"  [{idx}] {name:20s} -> {count:,} tweets ({count/len(df)*100:.1f}%)")

## Parte 3: Modelos Pre - entrenados

In [ ]:
# ============================================================
# Comparativa de modelos preentrenados en español
# ============================================================
muestras       = df[["text_clean", SENTIMENT_COL]].sample(5, random_state=SEED)
tweets_demo    = muestras["text_clean"].tolist()
etiquetas_real = muestras[SENTIMENT_COL].tolist()

print("\nTweets de prueba:")
for i, (t, e) in enumerate(zip(tweets_demo, etiquetas_real)):
    print(f"  [{i+1}] Real: {e:10s} | {t[:70]}...")

modelos_comparar = {
    "BETO (Wikipedia)":              "dccuchile/bert-base-spanish-wwm-cased",
    "RoBERTuito (500M tweets ES)":   "pysentimiento/robertuito-base-uncased",
    "BETO-Sentiment (tweets ES)":    "finiteautomata/beto-sentiment-analysis",
    "RoBERTuito-Sentiment":          "pysentimiento/robertuito-sentiment-analysis",
}

resultados_modelos = {}
for nombre, model_id in modelos_comparar.items():
    print(f"\nProbando: {nombre}")
    try:
        clf_tmp = pipeline(
            "text-classification", model=model_id, tokenizer=model_id,
            truncation=True, max_length=128,
            device=0 if torch.cuda.is_available() else -1,
        )
        preds = []
        for tweet in tweets_demo:
            res = clf_tmp(tweet)[0]
            preds.append(f"{res['label']} ({res['score']:.2f})")
        resultados_modelos[nombre] = preds
        print("  OK")
    except Exception as e:
        resultados_modelos[nombre] = [f"Error: {str(e)[:40]}"] * len(tweets_demo)
        print(f"  Error: {str(e)[:60]}")

df_comp = pd.DataFrame(resultados_modelos)
df_comp.insert(0, "Etiqueta Real", etiquetas_real)
df_comp.insert(1, "Tweet", [t[:40] + "..." for t in tweets_demo])

print("\nComparativa de modelos:")
print(df_comp.to_string(index=False))

In [ ]:
# ============================================================
# Division Train / Validation / Test (70 / 15 / 15)
# ============================================================
X = df["text_clean"].tolist()
y = df["label"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f"Train:      {len(X_train):>6,}")
print(f"Validation: {len(X_val):>6,}")
print(f"Test:       {len(X_test):>6,}")

## Parte 4: Beto y Tokenización

In [ ]:
# ============================================================
# Carga del tokenizer (BETO)
# ============================================================
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verificacion rapida
emoji_test = "Me encanta este producto! :)"
tokens = tokenizer.tokenize(emoji_test)
print(f"\nTexto:  {emoji_test}")
print(f"Tokens: {tokens}")

In [ ]:
# ============================================================
# Pre-tokenizacion Eficiente con libreria 'datasets'
# ============================================================
MAX_LENGTH = 128

# 1. Convertimos los splits a diccionarios compatibles con Hugging Face
train_hf = Dataset.from_dict({"text": X_train, "labels": y_train})
val_hf   = Dataset.from_dict({"text": X_val,   "labels": y_val})
test_hf  = Dataset.from_dict({"text": X_test,  "labels": y_test})

# 2. Funcion de tokenizacion map
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

print("\nTokenizando datasets...")
# 3. Aplicar tokenizacion en lotes (mucho mas rapido)
train_dataset = train_hf.map(tokenize_fn, batched=True)
val_dataset   = val_hf.map(tokenize_fn, batched=True)
test_dataset  = test_hf.map(tokenize_fn, batched=True)

# 4. Formato final para PyTorch
cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=cols_to_keep)
val_dataset.set_format(type="torch", columns=cols_to_keep)
test_dataset.set_format(type="torch", columns=cols_to_keep)

# Validacion
muestra = train_dataset[0]
print(f"input_ids shape:      {muestra['input_ids'].shape}")
print(f"attention_mask shape: {muestra['attention_mask'].shape}")
print(f"label:                {muestra['labels'].item()} ({id2label[muestra['labels'].item()]})")

## Parte 5: Pre - Evaluación

In [ ]:
# ============================================================
# Funciones de evaluacion
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="macro")

    return {"accuracy": acc, "f1_macro": f1}


def reporte_completo(y_real, y_pred, titulo="Reporte"):
    print(f"\n{'=' * 60}")
    print(f"  {titulo}")
    print(f"{'=' * 60}")
    print(classification_report(y_real, y_pred, target_names=LABEL_NAMES, digits=4))

    cm = confusion_matrix(y_real, y_pred)
    plt.figure(figsize=(max(6, NUM_LABELS * 1.5), max(5, NUM_LABELS * 1.2)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    plt.title(f"Matriz de confusion - {titulo}", fontsize=13, fontweight="bold")
    plt.ylabel("Real")
    plt.xlabel("Prediccion")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"confusion_{titulo.replace(' ', '_')}.png", dpi=150)
    plt.show()


In [ ]:
# ============================================================
# Evaluacion ANTES del fine-tuning
# ============================================================
model_pre = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)
model_pre.to(device)
model_pre.eval()

def get_eval_args(output_dir):
    kwargs = dict(output_dir=output_dir, per_device_eval_batch_size=32, report_to="none")
    if pv.parse(transformers.__version__) >= pv.parse("4.32.0"):
        kwargs["use_cpu"] = (device == "cpu")
    else:
        kwargs["no_cuda"] = (device == "cpu")
    return TrainingArguments(**kwargs)

trainer_pre = Trainer(
    model=model_pre, args=get_eval_args("./pre_eval"),
    eval_dataset=test_dataset, compute_metrics=compute_metrics,
)

print("\nEvaluando modelo ANTES del fine-tuning...")
pre_metrics  = trainer_pre.evaluate()
pre_accuracy = pre_metrics["eval_accuracy"]
pre_f1       = pre_metrics["eval_f1_macro"]
print(f"Accuracy pre fine-tuning: {pre_accuracy:.4f} | F1 Macro: {pre_f1:.4f}")

pre_preds = np.argmax(trainer_pre.predict(test_dataset).predictions, axis=-1)
reporte_completo(y_test, pre_preds, titulo="ANTES del Fine-Tuning")

## Parte 6: Fine Tuning

In [ ]:
# ============================================================
# Fine-tuning
# ============================================================
EPOCHS        = 8
BATCH_TRAIN   = 8
BATCH_EVAL    = 32
LEARNING_RATE = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.15
GRAD_ACCUM    = 4
MAX_GRAD_NORM = 1.0

usar_fp16 = torch.cuda.is_available()

model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2,
)

n_params = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parametros entrenables: {n_params:,} ({n_params/1e6:.1f}M)")

training_args = TrainingArguments(
    output_dir                  = "./bert_tweets_es",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",
    max_grad_norm               = MAX_GRAD_NORM,
    fp16                        = usar_fp16,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    greater_is_better           = True,
    save_total_limit            = 2,
    logging_dir                 = "./logs",
    logging_steps               = 30,
    report_to                   = "none",
    seed                        = SEED,
)

trainer_ft = Trainer(
    model=model_ft, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3,
                                     early_stopping_threshold=0.005)],
)

print("\nIniciando fine-tuning...\n")
train_result = trainer_ft.train()

print(f"\nFine-tuning completado")
print(f"  Tiempo: {train_result.metrics['train_runtime']:.1f}s "
      f"({train_result.metrics['train_runtime']/60:.1f} min)")
print(f"  Loss final: {train_result.metrics['train_loss']:.4f}")

In [ ]:
# ============================================================
# Curvas de entrenamiento
# ============================================================
log_history = trainer_ft.state.log_history

train_losses = [(e["epoch"], e["loss"])
                for e in log_history if "loss" in e and "eval_loss" not in e]
eval_metrics = [(e["epoch"], e["eval_loss"], e["eval_f1_macro"])
                for e in log_history if "eval_loss" in e]

if train_losses and eval_metrics:
    epochs_tr, losses_tr = zip(*train_losses)
    epochs_ev, losses_ev, f1s_ev = zip(*eval_metrics)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_tr, losses_tr, label="Train Loss", color="#E74C3C", linewidth=1.5)
    axes[0].plot(epochs_ev, losses_ev, "o--", label="Val Loss",
                 color="#2980B9", linewidth=1.5, markersize=8)
    axes[0].set_title("Perdida durante el entrenamiento", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs_ev, f1s_ev, "s-", color="#27AE60", linewidth=2, markersize=10)
    axes[1].set_title("F1-Score (Macro) de Validacion por Epoca", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("F1-Score")
    axes[1].set_ylim(0, 1)
    for e, f1 in zip(epochs_ev, f1s_ev):
        axes[1].annotate(f"{f1:.3f}", (e, f1), textcoords="offset points",
                         xytext=(0, 10), ha="center", fontsize=10, fontweight="bold")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Historial insuficiente para graficar.")

In [ ]:
# ============================================================
# Evaluacion DESPUES del fine-tuning
# ============================================================
print("\nEvaluando modelo DESPUES del fine-tuning...")

post_output   = trainer_ft.predict(test_dataset)
post_preds    = np.argmax(post_output.predictions, axis=-1)
post_accuracy = accuracy_score(y_test, post_preds)
post_f1       = f1_score(y_test, post_preds, average="macro")

reporte_completo(y_test, post_preds, titulo="DESPUES del Fine-Tuning")

print(f"\n{'=' * 60}")
print(f"  RESUMEN COMPARATIVO")
print(f"{'=' * 60}")
print(f"  Accuracy ANTES:   {pre_accuracy:.4f}  |  DESPUES: {post_accuracy:.4f}  (Mejora: {post_accuracy - pre_accuracy:+.4f})")
print(f"  F1-Score ANTES:   {pre_f1:.4f}  |  DESPUES: {post_f1:.4f}  (Mejora: {post_f1 - pre_f1:+.4f})")
print(f"{'=' * 60}")

## Parte 7: Comparativo e inferencia

In [ ]:
# ============================================================
# Grafico comparativo antes vs despues
# ============================================================
fig, ax = plt.subplots(figsize=(7, 5))

etiquetas_bar = ["ANTES\nFine-Tuning", "DESPUES\nFine-Tuning"]
colores       = ["#E74C3C", "#27AE60"]

bars = ax.bar(etiquetas_bar, [pre_accuracy, post_accuracy],
              color=colores, width=0.4, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, [pre_accuracy, post_accuracy]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val*100:.2f}%", ha="center", va="bottom",
            fontsize=14, fontweight="bold")

ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title(f"Impacto del fine-tuning en BETO\n({MODEL_NAME})",
             fontsize=13, fontweight="bold")
ax.axhline(y=1/NUM_LABELS, color="gray", linestyle="--", alpha=0.7,
           label=f"Baseline aleatorio ({1/NUM_LABELS:.2%})")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("comparativo_antes_despues.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# Inferencia sobre tweets nuevos
# ============================================================
clf_pipeline = pipeline(
    "text-classification", model=trainer_ft.model, tokenizer=tokenizer,
    device=0 if device == "cuda" else -1,
    truncation=True, max_length=MAX_LENGTH,
)

tweets_nuevos = [
    "Estoy tan feliz hoy, todo salio perfecto",
    "Me siento muy triste y agotado, no puedo mas",
    "Que dia tan estresante, estoy completamente desbordado @jefe",
    "Simplemente un dia normal, sin nada especial",
    "INCREIBLE! No puedo creer lo que acaba de pasar",
    "Estoy asustado de lo que viene, el futuro es incierto",
    "Me siento muy abrumado por tanto trabajo",
    "Dios mio no puedo creer esta noticia http://noticias.com",
    "Estoy tan feliz, todo del orto",
    "Esto es una mierda"
]

print(f"\n{'Sentimiento':>15s}  {'Conf':>5s}  Tweet")
print("-" * 70)
for tweet in tweets_nuevos:
    tweet_limpio = limpiar_tweet(tweet)
    resultado = clf_pipeline(tweet_limpio)[0]

    label_raw = resultado["label"]
    if label_raw.startswith("LABEL_"):
      label = id2label[int(label_raw.split("_")[1])]
    else:
      label = label_raw
    conf = resultado["score"]
    print(f"{label:>15s}  {conf:.3f}  {tweet}")

## Parte 8: Conclusiones

1. Efectividad contundente del Transfer Learning

* El experimento demuestra empíricamente el valor del fine-tuning sobre modelos preentrenados. Antes del entrenamiento, la capa de clasificación (inicializada aleatoriamente) arrojaba un Accuracy del 13.11%, un valor inferior incluso al baseline de probabilidad aleatoria pura (16.67% para 6 clases). Tras solo 8 épocas de adaptación, el modelo alcanzó un Accuracy del 70.44% y un F1-Score (Macro) del 70.25%, logrando un incremento absoluto de casi 60 puntos porcentuales. Esto confirma que los pesos lingüísticos generales de BETO se adaptaron con éxito a la semántica específica del análisis de sentimientos.

2. Robustez frente al desbalance de clases:

* El Análisis Exploratorio (EDA) reveló cierto desbalance en el dataset, siendo Peaceful la clase mayoritaria y Scared la minoritaria. Sin embargo, la paridad obtenida entre el Accuracy (70.44%) y el F1-Score Macro (70.25%) indica que el modelo no desarrolló un sesgo hacia la clase mayoritaria.
* Al observar la matriz de confusión final, se aprecia una diagonal fuertemente marcada. Los errores cometidos por el modelo tienen "sentido semántico"; por ejemplo, existe cierta confusión entre Peaceful y Powerful, o entre Sad y Mad.
* Al ser emociones adyacentes o coexistentes en un mismo texto, estos falsos positivos demuestran que el modelo aprendió representaciones latentes coherentes y no está fallando de manera arbitraria.

3. Convergencia y Estabilidad del Entrenamiento

* Las curvas de entrenamiento reflejan un proceso de optimización muy estable. La pérdida de validación (Val Loss) desciende drásticamente durante las primeras 3 épocas y luego se estabiliza alrededor de 0.9, mientras que el F1-Score de validación crece progresivamente hasta alcanzar un plateau (meseta) en la época 6 y 7 (~0.740).
* La ausencia de una divergencia en la curva de Val Loss confirma que el modelo no sufrió de un sobreajuste (overfitting) severo, validando la elección de hiperparámetros como el Weight Decay (0.01), el Dropout (0.2) y el Early Stopping.

4. Comprensión semántica y manejo de modismos

Las pruebas de inferencia sobre "tweets nuevos" confirman la capacidad de generalización del modelo en un entorno real:

* Comprensión de polaridad y jerga: Clasifica de manera certera expresiones fuertemente polarizadas ("Esto es una mierda" → Sad / "INCREIBLE!" → Powerful).
* Gestión de ambigüedad: En el tweet "Estoy tan feliz, todo del orto", el modelo se inclinó por Peaceful (0.678), dándole mayor peso semántico a la palabra "feliz" por encima del modismo regional
.
5. Posibles líneas de trabajo futuro

Aunque los resultados son muy satisfactorios para un modelo fundacional genérico como BETO, el rendimiento (70%) deja margen de mejora. Como trabajo futuro se sugiere:

* Experimentar utilizando como base un modelo preentrenado nativamente en redes sociales, como RoBERTuito, el cual ya posee embeddings específicos para la sintaxis y gramática informal de Twitter (X).
* Aplicar técnicas de Data Augmentation (como back-translation o parafraseo) para aumentar el volumen de la clase minoritaria (Scared).